# Lab 3.3 &mdash; Conditional Routing

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; LangGraph: Stateful Agent Workflows**

### What you'll do
- Write a routing function &mdash; state in, the name of the next node out
- Attach it with <code>add_conditional_edges</code> and a branch map
- Send three requests down three paths through one graph

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All eight Module 3 labs work one case: leave requests in a small HR
> system. The rules are ordinary on purpose &mdash; the only new thing here is LangGraph.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# Leave requests in a small HR system. Ordinary rules on purpose: the only new thing in these
# eight labs is LangGraph. One flat dict -- no joins, no helpers, nothing to learn here.

REQUESTS = {
    "LV-5001": {"who": "Priya Nair",   "days":  3, "kind": "annual", "reason": "family wedding",
                "balance": 12, "manager": "Devi R."},
    "LV-5002": {"who": "Rahul Menon",  "days":  5, "kind": "annual", "reason": "",
                "balance":  3, "manager": "Devi R."},
    "LV-5003": {"who": "Anita Sharma", "days":  2, "kind": "annual", "reason": "moving house",
                "balance":  0, "manager": "Sam O."},
    "LV-5004": {"who": "Vikram Rao",   "days": 15, "kind": "annual", "reason": "sabbatical",
                "balance": 20, "manager": "Sam O."},
    "LV-5005": {"who": "Priya Nair",   "days":  1, "kind": "sick",   "reason": "flu",
                "balance": 12, "manager": "Devi R."},
}

# The handbook, as three numbers. Every routing decision in this module comes from these.
POLICY = {"manager_over_days": 2, "hr_over_days": 10, "max_clarifications": 2}

print(len(REQUESTS), "leave requests loaded")

## Concept

Every edge so far was unconditional: after A, always B. A **conditional edge** asks first.

```python
builder.add_conditional_edges("check", route, {"auto": "approve", "no": "decline"})
```

Three arguments: the node the decision happens **after**, a routing function, and a **branch map**
from the strings the router returns to the nodes they mean.

Two things to get right from the start:

- The router **reads state and returns a string**. It does no work, calls no model, changes
  nothing. Keep it that way and your routing stays unit-testable.
- It hangs off **the node that produced the facts it reads**. Attach it earlier and it decides on
  fields that do not exist yet.

Module 5's supervisor is this function with a model choosing the string. You are writing one now.

## Section 1 &mdash; The router

The `if`/`elif` is written for you. The decision is which branch each situation belongs to &mdash;
and the order, because someone with no balance left should be declined *whether or not* the
request is long enough to need a manager.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class LeaveState(TypedDict):
    request_id: str
    days: int
    balance: int
    covered: bool
    needs_manager: bool
    decision: str
    notes: list


def route(state: LeaveState) -> str:
    """Which node runs next? Returns a branch name and nothing else."""
    if not state["covered"]:
        return BLANK       # TODO: no balance to cover it -- which branch?
    if state["needs_manager"]:
        return BLANK       # TODO: covered, but too long to decide alone -- which branch?
    return "auto"

In [ ]:
# --- Self-check: Section 1   (a pure function, so this is four dicts)
check("no balance -> decline, whatever else is true",
      lambda: route({"covered": False, "needs_manager": False}) == "decline"
          and route({"covered": False, "needs_manager": True}) == "decline",
      "there is nothing for a manager to approve if the balance is not there")
check("covered and long -> manager; covered and short -> auto",
      lambda: route({"covered": True, "needs_manager": True}) == "manager"
          and route({"covered": True, "needs_manager": False}) == "auto")
score()

## Section 2 &mdash; Attaching it

The branch map is why the router can return a short name like `"auto"` without knowing what the
node is called &mdash; routing logic and graph layout stay separable.

In [ ]:
def summarise(state):
    r = REQUESTS[state["request_id"]]
    return {"days": r["days"], "balance": r["balance"], "notes": state["notes"] + ["summarise"]}

def check_request(state):
    return {"covered": state["balance"] >= state["days"],
            "needs_manager": state["days"] > POLICY["manager_over_days"],
            "notes": state["notes"] + ["check"]}

def approve(state):  return {"decision": "approved automatically",  "notes": state["notes"] + ["approve"]}
def to_manager(s):   return {"decision": f'queued for {REQUESTS[s["request_id"]]["manager"]}',
                             "notes": s["notes"] + ["to_manager"]}
def decline(state):  return {"decision": "declined: not enough balance", "notes": state["notes"] + ["decline"]}


def build_router_graph():
    builder = StateGraph(LeaveState)
    for name, fn in [("summarise", summarise), ("check", check_request), ("approve", approve),
                     ("to_manager", to_manager), ("decline", decline)]:
        builder.add_node(name, fn)

    builder.add_edge(START, "summarise")
    builder.add_edge("summarise", "check")

    builder.add_conditional_edges(
        BLANK,                                   # TODO: after WHICH node is the decision made?
        route,
        {"auto": "approve", "manager": "to_manager", "decline": "decline"},
    )

    for name in ["approve", "to_manager", "decline"]:
        builder.add_edge(name, END)
    return builder.compile()

In [ ]:
# --- Self-check: Section 2   (one real graph, three real paths, no model)
def run(rid):
    return build_router_graph().invoke({"request_id": rid, "notes": []})

check("three requests take three different branches",
      lambda: (run("LV-5005")["notes"][-1], run("LV-5004")["notes"][-1],
               run("LV-5003")["notes"][-1]) == ("approve", "to_manager", "decline"))
check("only ONE branch runs per request",
      lambda: len({"approve", "to_manager", "decline"} & set(run("LV-5004")["notes"])) == 1,
      "a conditional edge chooses one target; it does not run them all")
score()

## Watch it run

In [ ]:
app = guard(build_router_graph)

if app is not None:
    print(f"{'request':10}{'days':>5}{'bal':>5}   {'path':38} decision")
    print("-" * 100)
    for rid in sorted(REQUESTS):
        out = app.invoke({"request_id": rid, "notes": []})
        print(f"{rid:10}{out['days']:5}{out['balance']:5}   "
              f"{' -> '.join(out['notes']):38} {out['decision']}")

### Read it

Every row went through `summarise` and `check`, then diverged. That divergence is the first thing
in this module you could not have written as a chain.

The router stayed a pure function, which is why the hardest logic in the workflow is also the
cheapest thing to test &mdash; four dicts, no graph.

In [ ]:
score()

## Your turn

1. `POLICY["hr_over_days"]` is 10 and nothing uses it. Add an `hr_queue` node and a fourth branch.
   Notice you change the router and the map, and touch no existing node.
2. Make `route` return a name that is not in the branch map and read the error. That is the
   failure mode of an LLM-routed supervisor in Module 5, met under controlled conditions.